In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.functions import (col, date_trunc, md5, current_timestamp)
from datetime import datetime

# Widget with default value
dbutils.widgets.text("table_name", "dim_location")
table_name = dbutils.widgets.get("table_name")

# Configuration & Paths
silver_base = "abfss://processed@[YOUR_STORAGE_ACCOUNT_NAME].dfs.core.windows.net/silver/"
gold_base = "abfss://curated@[YOUR_STORAGE_ACCOUNT_NAME].dfs.core.windows.net/gold/"
target_table = table_name
gold_path = f"{gold_base}{target_table}"

# Silver Sources
df_geo = spark.read.format("delta").load(f"{silver_base}olist_geolocation_dataset")
df_state = spark.read.format("delta").load(f"{gold_base}dim_states")

# Transformation Logic
df_dim_base = (df_geo.alias("g")
               .join(df_state.alias("st"), F.col("g.geolocation_state") == F.col("st.state_code"), "left")
               .select(
                   F.md5(F.col("geolocation_zip_code_prefix")).cast("string").alias("location_key"),
                   "g.geolocation_zip_code_prefix",
                   "g.geolocation_city",
                   F.col("g.geolocation_state").alias("state_code"),
                   "st.state_name",
                   "st.region",
                   F.col("g.geolocation_lat").alias("latitude"),
                   F.col("g.geolocation_lng").alias("longitude"),
                   F.date_trunc("second", F.current_timestamp()).alias("gold_load_at")
               ))

# Unknown record
now = datetime.now().replace(microsecond=0)
unknown_data = [(-1, None, "NA", "NA", "NA", "NA", None, None, now)]
target_schema = df_dim_base.schema
df_unknown = spark.createDataFrame(unknown_data, schema=target_schema)
df_final = df_dim_base.unionByName(df_unknown)

# Incremental Merge (Upsert)
print(f"--> Starting Gold Load: {target_table}")

if DeltaTable.isDeltaTable(spark, gold_path):
    dt_gold = DeltaTable.forPath(spark, gold_path)

    (dt_gold.alias("target")
     .merge(df_final.alias("source"), "target.geolocation_zip_code_prefix = source.geolocation_zip_code_prefix")
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .execute())
    
    print(f"--> [Merge] {target_table} updated.")
else:
    #First time load
    df_final.write.format("delta").mode("overwrite").save(gold_path)
    print(f"--> [INIT] {target_table} created.")

# Audit & Exit
dt_final = DeltaTable.forPath(spark, gold_path)

last_operation = dt_final.history(1).select("operationMetrics").collect()[0][0]
rows_inserted = int(last_operation.get("numTargetRowsInserted", 0))
rows_updated = int(last_operation.get("numTargetRowsUpdated", 0))
total_rows = last_operation.get("numOutputRows", "Check History")

print("-" * 50)
print(f"--> Table: {table_name} | Status: Success")
print(f"--> Rows Processed in last run: {rows_inserted + rows_updated}")
print("-" * 50)

dbutils.notebook.exit(f"Success | Inserted: {rows_inserted} | Updated: {rows_updated}")